In [ ]:
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git
!pip install image-classifiers


In [ ]:
import os
import pandas as pd
from huggingface_hub import snapshot_download
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from huggingface_hub import hf_hub_download
import random
import warnings

from sklearn.manifold import TSNE
import numpy as np
import cv2


import seaborn as sns
import matplotlib.pyplot as plt

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, regularizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from classification_models.tfkeras import Classifiers

# Machine Learning Utility (Scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight

warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image
from sklearn.neighbors import NearestNeighbors
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

In [ ]:
ROOT_DIR = "/content/GroceryStoreDataset/dataset/" # The base directory where the images and CSVs are stored
# Loads the 'classes.csv' of the dataset metadata
classes_df = pd.read_csv(os.path.join(ROOT_DIR, "classes.csv"))
print(f"Shape: {classes_df.shape}")
print(f"Unique Classes: {classes_df['Class ID (int)'].unique()}") # Checks how many distinct categories exist
classes_df.head() # Shows the first 5 rows of the pandas dataframe of the metadata

In [ ]:
def process_dataframe_fixed(df, root_dir): #creare the right path for the data loading
    df['path'] = df['path'].apply(lambda x: os.path.join(root_dir, x)) #directly inside the dataframe pandas
    df['fine_label'] = df['fine_label'].astype(str)
    return df

#create all the dataset
train_df = pd.read_csv(os.path.join(ROOT_DIR, "train.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])
val_df   = pd.read_csv(os.path.join(ROOT_DIR, "val.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])
test_df  = pd.read_csv(os.path.join(ROOT_DIR, "test.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])
print(test_df.head(1))
#keep only final_label
train_df.drop(columns=["coarse_label"], inplace=True)
val_df.drop(columns=["coarse_label"], inplace=True)
test_df.drop(columns=["coarse_label"], inplace=True)
print(test_df.head(1))
train_df = process_dataframe_fixed(train_df, ROOT_DIR)
val_df = process_dataframe_fixed(val_df, ROOT_DIR)
test_df = process_dataframe_fixed(test_df, ROOT_DIR)

print(f"Labels class in Train: {train_df['fine_label'].nunique()}")
print(f"Labels class in Val:   {val_df['fine_label'].nunique()}")
print(f"Labeles class in Test:  {test_df['fine_label'].nunique()}")



In [ ]:
df_combined = pd.concat([train_df, val_df], ignore_index=True) #merge the two dataset val and train because i want recreate the entire dataset
#lista_classi = sorted(df_combined['fine_label'].unique().tolist()) #create a list of the classes
#NUM_CLASSES = len(lista_classi)
train_df_new, val_df_new = train_test_split(
    df_combined,
    test_size=0.20, #val is 20%
    stratify=df_combined['fine_label'], # keep the same class label distribution between val and train
    random_state=42
)

In [ ]:
repo_path = snapshot_download(
    repo_id="tomasconti/Drift_Detection",
    repo_type="model"   # or "dataset" if applicable
)
print(repo_path)

In [ ]:
new_classes = { "Eags-20260608T191746Z-3-001/Eags": "81",
               "Still_water_images-20260608T193520Z-3-001/Still_water_images": "82",
                "Yogurt -20260608T193254Z-3-001/Yogurt ": "83",}
rows = []
for folder, label in new_classes.items():
    folder_path = Path(repo_path) / folder
    if not folder_path.exists():
        print(f"Folder not found: {folder_path}")
        continue
    for img in folder_path.rglob("*"):
        if img.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            rows.append({"path": str(img),"fine_label": label})

new_df = pd.DataFrame(rows)
new_df["fine_label"] = new_df["fine_label"].astype(str)
new_train, temp = train_test_split( new_df, test_size=0.5, stratify=new_df["fine_label"],random_state=42,) #50% for train
new_val, new_test = train_test_split(temp,test_size=0.6,stratify=temp["fine_label"],random_state=42,) # 30% for test and 20% for val

print("Train:", len(new_train))
print("Val:", len(new_val))
print("Test:", len(new_test))

print("\nTrain distribution")
print(new_train["fine_label"].value_counts())

print("\nValidation distribution")
print(new_val["fine_label"].value_counts())

print("\nTest distribution")
print(new_test["fine_label"].value_counts())

train_df = pd.concat([train_df_new, new_train], ignore_index=True)
val_df   = pd.concat([val_df_new, new_val], ignore_index=True)
test_df  = pd.concat([test_df, new_test], ignore_index=True)

print("\nFinal datasets")
print("Train:", len(train_df), "images")
print("Val:", len(val_df), "images")
print("Test:", len(test_df), "images")

print("\nClasses")
print("Train:", train_df["fine_label"].nunique())
print("Val:", val_df["fine_label"].nunique())
print("Test:", test_df["fine_label"].nunique())

In [ ]:
lista_classi = sorted(train_df['fine_label'].unique().tolist())
IMG_RAW_H, IMG_RAW_W = 256, 256
# Target dimensions after internal cropping (matches standard ResNet input)
IMG_CROP_H, IMG_CROP_W = 224, 224
CHANNELS = 3
BATCH_SIZE = 64
EPOCHS = 60
# Loading ResNet18 classifier and its specific preprocessing requirements.
ResNet18, preprocess_input = Classifiers.get('resnet18')
# use the specific ResNet preprocessing function
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='fine_label',
    target_size=(IMG_RAW_H, IMG_RAW_W),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=lista_classi,
    shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='path',
    y_col='fine_label',
    target_size=(IMG_RAW_H, IMG_RAW_W),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=lista_classi,
    shuffle=False
)

train_labels = train_generator.classes
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
dict_weights = dict(enumerate(weights))

In [ ]:
NUM_CLASSES = len(lista_classi)
print(NUM_CLASSES)

In [ ]:
def build_resnet18_advanced(num_classes, learning_rate, fine_tune=False):
    """
    Constructs an advanced ResNet18 model using Transfer Learning.
    Args:
        num_classes: Number of target categories.
        learning_rate: Step size for the optimizer.
        fine_tune: If True, unlocks the base model weights for updating.
    """
    # --- 1. INPUT LAYER ---
    # Receives the high-resolution raw images (e.g., 256x256x3)
    inputs = layers.Input(shape=(IMG_RAW_H, IMG_RAW_W, 3), name="input_image")
    # --- 2. GPU-ACCELERATED DATA AUGMENTATION ---
    # These layers are active only during training for the generalization
    x = layers.RandomCrop(IMG_CROP_H, IMG_CROP_W)(inputs) # Improves scale invariance
    x = layers.RandomFlip("horizontal")(x)                # Improves symmetry robustness
    x = layers.RandomRotation(0.2)(x)                     # Handles orientation variance
    x = layers.RandomContrast(0.1)(x)                     # Robustness to lighting/exposure
    x = layers.RandomTranslation(0.1, 0.1)(x)             # Invariance to object positioning

    # --- 3. PRE-TRAINED BASE MODEL (Backbone) ---
    # Load ResNet18 with weights pre-trained on ImageNet.
    # include_top=False: Drops the original 1000-class classification head.
    # input_tensor=x: create a pipe from data agumentation and model backbone
    base_model = ResNet18(
        weights='imagenet',
        include_top=False,
        input_tensor=x
    )
    # fine_tune=False: 'Feature Extraction' mode (base weights are frozen)
    # fine_tune=True:  'Fine-Tuning' mode (base weights will be updated)
    base_model.trainable = fine_tune
    # --- 5. CUSTOM CLASSIFICATION HEAD ---
    # Extract the high-level semantic features from the backbone.
    x = base_model.output
    #the head architecture is the same from the first task for better comparision
    x = layers.GlobalAveragePooling2D()(x)
    embedding = layers.Dense( 512, activation=None, kernel_regularizer=regularizers.l2(0.005),name="embedding_layer" )(x)
    #smallest regularization
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001))(embedding)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="ResNet18_TransferLearning")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    return model

In [ ]:
#callbacks designed for the Fine-tuning phase to handle convergence
callbacks_list = [
    # early stopping monitor validation loss to halt training when the model
    # stops generalizing, preventing overfitting
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=12,            # Sufficient window to allow Learning Rate decay to take effect
        restore_best_weights=True, # Reverts to the epoch with the lowest validation loss
        verbose=1
    ),
    # reduces the  learning rate when progress stalls to settle into a sharper local minimum
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', #if the model does not see any loss improvement i don't learn nothing, model stock
        factor=0.5,   # LR new = LR × 0.5 (50% reduction), looking for the local mini valley
        patience=4,             # wait 4 epochs of stagnation before cutting the LR
        min_lr=1e-7,            # Lower bound to prevent the LR from vanishing
        verbose=1
    )
]

# clear the gpu session (dram)
tf.keras.backend.clear_session()

#  warm-up: freeze ResNet backbone, train only the new classification head
# head weight stabilization
print("\n Warm-up")
# fine_tune=False: Locks the pre-trained weights.
# A moderate LR (0.0004) is safe here as we are only training a few layers.
model2 = build_resnet18_advanced(NUM_CLASSES, learning_rate=0.0004, fine_tune=False)

history_warmup = model2.fit(
    train_generator,
    epochs=5,               # Short burst: just enough to reach a stable starting point
    validation_data=val_generator,
    class_weight=dict_weights,
    callbacks=[
        # Phase-specific EarlyStopping: stop as soon as the head stops improving quickly
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=3,
            restore_best_weights=True,
            verbose=1
        )
    ]
)

#Unfreeze the entirSe network and train all the parameter to tune the system
print("\nFine-tuning")
# reuse weights from the warmed-up model to use as a baseline
weights_warmup = model2.get_weights()

# small learning rate because we are looking for a parameter tuning, i want keep the feature extraction knowladge
model2 = build_resnet18_advanced(NUM_CLASSES, learning_rate=0.0001, fine_tune=True)
# inject the weights from the warm up into the new unfrozen model architecture
model2.set_weights(weights_warmup)
history_finetune = model2.fit(
    train_generator,
    epochs=55,              # Extensive budget; EarlyStopping< will handle the exit
    validation_data=val_generator,
    class_weight=dict_weights,
    callbacks=callbacks_list # Full callback suite for a safe and robust training run
)

In [ ]:
def plot_history(history, title="Training History"):
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(title + " - Loss")
    plt.legend()
    plt.show()

plot_history(history_warmup, title="Warm-up")
plot_history(history_finetune, title="Fine-tuning")


In [ ]:
test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='fine_label',
    #y_col='coarse_label',
    target_size=(IMG_RAW_H, IMG_RAW_W),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=lista_classi,
    shuffle=False
)

In [ ]:
test_loss, test_accuracy = model2.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
model2.save("resnet18_84C_embedding_layer_v1.keras")

In [ ]:
embedding_model = tf.keras.Model( inputs=model2.input, outputs=model2.get_layer("embedding_layer").output )

In [ ]:
train_embeddings_84C = embedding_model.predict(train_generator)
print(train_embeddings_84C.shape)
val_embeddings_84C = embedding_model.predict(val_generator)
print(val_embeddings_84C.shape)
test_embeddings_84C = embedding_model.predict(test_generator)
print(test_embeddings_84C.shape)

In [ ]:
np.save("train_embeddings_84C.npy", train_embeddings_84C)
np.save("val_embeddings_84C.npy", val_embeddings_84C)
np.save("test_embeddings_84C.npy", test_embeddings_84C)
print("Embeddings saved!")

In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("Downloading native 512-D embeddings from HuggingFace")

path_train = hf_hub_download(repo_id=REPO_ID, filename="84C/train_embeddings_84C.npy")
path_val = hf_hub_download(repo_id=REPO_ID, filename="84C/val_embeddings_84C.npy")
path_test = hf_hub_download(repo_id=REPO_ID, filename="84C/test_embeddings_84C.npy")


test_embeddings_84C = np.load(path_test)
val_embeddings_84C = np.load(path_val)
train_embeddings_84C = np.load(path_train)


In [ ]:
train_labels = train_generator.classes
tsne = TSNE( n_components=2, perplexity=50, learning_rate="auto", init="pca", random_state=42 )
embeddings_2d = tsne.fit_transform(train_embeddings_84C)
print("t-SNE shape:", embeddings_2d.shape)
plt.figure(figsize=(12,10))
scatter = plt.scatter( embeddings_2d[:,0], embeddings_2d[:,1], )
plt.title("t-SNE visualization of test set embeddings")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

In [ ]:
results = []
for linkage in ['ward', 'complete', 'average', 'single']:
    for k in range(75, 91):
        model = AgglomerativeClustering( n_clusters=k,linkage=linkage)
        labels = model.fit_predict(train_embeddings_84C)
        results.append({ "Linkage": linkage,"K": k,"Silhouette": silhouette_score(train_embeddings_84C, labels)})
df_results = pd.DataFrame(results)
plt.figure(figsize=(10,5))
sns.lineplot( data=df_results, x="K", y="Silhouette", hue="Linkage", marker="o")
plt.title("Agglomerative Clustering of embeddings")
plt.xlabel("Clusters")
plt.ylabel("Silhouette Score")
plt.grid()
plt.legend()
plt.show()

In [ ]:
n_clusters = 84
cluster_model = AgglomerativeClustering( n_clusters=n_clusters, linkage='ward')
labels = cluster_model.fit_predict(train_embeddings_84C)
centroids = np.array([ train_embeddings_84C[labels == i].mean(axis=0) for i in range(n_clusters)])
print("Centroids shape:", centroids.shape)

In [ ]:
all_data = np.concatenate( [train_embeddings_84C, centroids], axis=0 )
all_2d = TSNE( n_components=2, perplexity=50, random_state=42, init="pca").fit_transform(all_data)

train_2d = all_2d[:len(train_embeddings_84C)]
centroids_2d = all_2d[len(train_embeddings_84C):]

plt.figure(figsize=(12,8))
plt.scatter( train_2d[:,0], train_2d[:,1], s=15, alpha=0.4, label="Train embeddings" )

plt.scatter( centroids_2d[:,0], centroids_2d[:,1], marker="X", s=120, c="red", edgecolors="black", label="Cluster centroids" )

plt.title("t-SNE: Train embeddings + Agglomerative Centroids")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
knn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(centroids)
data_embeddings = { "Train": train_embeddings_84C,"Val": val_embeddings_84C, "Test": test_embeddings_84C }
distances = {
    name: knn.kneighbors(emb)[0].flatten()
    for name, emb in data_embeddings.items()
}

plt.figure(figsize=(10,6))

for name, dist in distances.items():
    sns.kdeplot(dist, fill=True, alpha=0.25, label=name)
    plt.axvline(np.median(dist), linestyle="--", label=f"{name} median")

plt.title("Distance to nearest centroid (K=1)")
plt.xlabel("Euclidean distance")
plt.ylabel("Density")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

df_dist = pd.DataFrame([
    {"Dataset": name, "Distance": d}
    for name, dist in distances.items()
    for d in dist
])

plt.figure(figsize=(10,5))
sns.boxplot(data=df_dist, x="Dataset", y="Distance")
plt.title("Distance distribution to nearest centroid")
plt.grid(axis="y", alpha=0.3)
plt.show()